Load the EuroVoc RDF file into an rdflib graph and extract all English scope notes for concepts with a eurovoc.europa.eu URI. Save the results to a JSON file for later use in the pipeline.

In [ ]:
from rdflib import Graph
from rdflib.namespace import SKOS
import json

# Load the RDF file
g = Graph()
g.parse("eurovoc_in_skos_core_concepts.rdf") 

print(f"Graph loaded: {len(g)} triples")

# Extract scope notes
scope_notes = {}
for concept, note in g.subject_objects(SKOS.scopeNote):
    uri = str(concept)
    if "eurovoc.europa.eu" in uri and note.language == "en":
        scope_notes[uri] = str(note)

print(f"Found {len(scope_notes)} English scope notes")

# Save to JSON
with open("eurovoc_scope_notes.json", "w") as f:
    json.dump(scope_notes, f, indent=2)

print("Saved to eurovoc_scope_notes.json")

Graph loaded: 520394 triples
Found 315 English scope notes
Saved to eurovoc_scope_notes.json


Extract the numeric concept IDs from the scope note URIs and verify they match the format used in eurovoc_descriptors.json.

In [3]:
import json

with open("eurovoc_scope_notes.json") as f:
    scope_notes = json.load(f)

# Extract the numeric IDs from the URIs
scope_note_ids = set(uri.split("/")[-1] for uri in scope_notes.keys())
print(f"Scope note concept IDs: {len(scope_note_ids)}")
print(list(scope_note_ids)[:10])  # inspect format

Scope note concept IDs: 315
['5778', '5776', '5772', '4607', '3540', '2117', '4479', '5887', '6311', '1789']




Check how many of the 5,000 MultiEURLEX English test documents have at least one gold label among the 315 concepts with English scope notes. This determines whether a scope note enrichment experiment is viable.

In [4]:
from datasets import load_dataset
import json

# Load your scope note IDs
with open("eurovoc_scope_notes.json") as f:
    scope_notes = json.load(f)

scope_note_ids = set(uri.split("/")[-1] for uri in scope_notes.keys())

# Load MultiEURLEX test set
dataset = load_dataset('coastalcph/multi_eurlex', 'en', 
                       label_level='all_levels', 
                       trust_remote_code=True)

test_set = dataset['test']

# Check coverage
docs_with_scope_note_label = 0
for doc in test_set:
    gold_labels = set(str(l) for l in doc['labels'])
    if gold_labels & scope_note_ids:  # intersection
        docs_with_scope_note_label += 1

print(f"Test documents with at least one scope-note label: {docs_with_scope_note_label} / {len(test_set)}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/multi_eurlex' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since coastalcph/multi_eurlex couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'en-label_level=all_levels' at /Users/vincent/.cache/huggingface/datasets/coastalcph___multi_eurlex/en-label_level=all_levels/1.0.0/5a12a7463045d4dcb12896b478c09b5a8a131a02b7e7bce059ba7ececc6584ee (last modified on Mon Feb  2 17:46:09 2026).


Test documents with at least one scope-note label: 520 / 5000


Count the number of scope notes available per language in the full EuroVoc RDF file. This reveals whether scope notes are available in the four target languages (English, French, Dutch, German) and could be used for native language label enrichment experiments.

In [6]:
from rdflib import Graph
from rdflib.namespace import SKOS

g = Graph()
g.parse("eurovoc_in_skos_core_concepts.rdf")

# Count scope notes by language
from collections import Counter
lang_counts = Counter()

for concept, note in g.subject_objects(SKOS.scopeNote):
    uri = str(concept)
    if "eurovoc.europa.eu" in uri:
        lang_counts[note.language] += 1

for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1]):
    print(f"{lang}: {count}")

lt: 449
es: 436
fr: 414
nl: 413
cs: 413
fi: 401
pt: 390
hu: 380
et: 369
da: 366
el: 364
sv: 359
de: 352
hr: 344
it: 333
en: 317
sk: 316
ro: 308
mt: 307
lv: 302
sq: 204
pl: 163
bg: 155
sl: 150
ga: 51
mk: 34
sr: 19
